# Single Chunk Debug Notebook

This notebook is a development sandbox for stepping through a **single chunk** of a textbook chapter. 
It uses the `%load` magic to pull in logic from `src/note_taker/`.

**Rules:**
- Do not write implementation code here. Write it in `src/`.
- Use `%load` to inspect and execute code step-by-step.
- Inspect the `GraphState` between nodes to debug the pipeline.

## Setup
Load environment variables, configure the development database, and enable autoreload.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import hashlib
from pathlib import Path
from rich import print as rprint

# Ensure src is in the path
sys.path.insert(0, str(Path().resolve().parent / 'src'))

from note_taker.database import DatabaseManager
from note_taker.processing import parse_markdown_chunks
from note_taker.pipeline.nodes import check_database_node
from note_taker.pipeline.state import GraphState

db = DatabaseManager('dev_notes.db')
db.ensure_database()

## Load Chapter File (SOLO-39)
Read the source Markdown file from disk.

In [ ]:
# Determine absolute path to the data directory
DATA_DIR = Path().resolve().parent / 'data' / 'raw' / 'building-applications-with'
target_file = DATA_DIR / 'chapter_005.md'

rprint(f"Target file exists: {target_file.exists()}")

## Chunk the Chapter (SOLO-39)
Split the loaded Markdown file into logical chunks (e.g., by header boundaries).

In [ ]:
# Chunk the chapter using the code-fence-aware splitter
# %load -s parse_markdown_chunks note_taker/processing.py

chunks = parse_markdown_chunks(str(target_file))
rprint(f"Total chunks found: [bold green]{len(chunks)}[/bold green]")

for i, c in enumerate(chunks):
    rprint(f"  [{i:03d}] {c['title']}")

## Pick a Single Chunk
Select one chunk from the list above by index to push through the pipeline.

In [ ]:
chunk_index = 8  # Change this to select a different chunk
selected_chunk = chunks[chunk_index]

rprint(f"Selected: [bold]{selected_chunk['title']}[/bold]")
rprint(f"Content length: {len(selected_chunk['content'])} chars")
rprint("---")
rprint(selected_chunk['content'][:300] + '...')

## Node: check_database (SOLO-38)
Check if this chunk has already been processed and its content hasn't changed. 
Sets `skip_processing` in `GraphState`.

In [ ]:
source_hash = hashlib.sha256(selected_chunk['content'].encode('utf-8')).hexdigest()
state: GraphState = {
    "chunk": selected_chunk['content'],
    "chunk_id": f"BuildingAIAgents:Chapter5:{chunk_index:03d}",
    "source_hash": source_hash,
    "force_refresh": False,
    "artifact": None,
    "skip_processing": False
}

new_state = check_database_node(state)
rprint(f"Chunk ID   : {state['chunk_id']}")
rprint(f"Source hash: {source_hash[:16]}...")
rprint(f"Skip processing? [bold]{'Yes' if new_state['skip_processing'] else 'No'}[/bold]")

## Node: draft (SOLO-41)
Generate the initial outline and Q&A pairs for the chunk.

In [ ]:
# TODO (SOLO-41): Wire up the draft node
# from note_taker.pipeline.nodes import draft_node
# draft_state = draft_node(new_state)
# rprint(draft_state['artifact'])

## Node: judge (SOLO-41)
Evaluate the drafted Q&A pairs for accuracy, clarity, and recall-worthiness.

In [ ]:
# TODO (SOLO-41): Wire up the judge node
# from note_taker.pipeline.nodes import judge_node
# judge_state = judge_node(draft_state)
# rprint(judge_state)

## Node: revise (SOLO-41)
Revise any questions that failed the judge's evaluation.

In [ ]:
# TODO (SOLO-41): Wire up the revise node
# from note_taker.pipeline.nodes import revise_node
# final_state = revise_node(judge_state)
# rprint(final_state['artifact'])

## Save to Dev DB (SOLO-38)
Write the final `FinalArtifactV1` to the development SQLite database.

In [ ]:
# TODO (SOLO-38): Save artifact after pipeline completes
# db.save_artifact(state['chunk_id'], state['source_hash'], final_state['artifact'])
# rprint('[green]Artifact saved to dev DB.[/green]')